# 01 · Extracción de datos en Google Earth Engine
**¿El fuego abre la puerta al cambio de uso del suelo en Chile?** — Premio MapBiomas Chile, 1.ª edición

Este notebook construye, para una muestra estratificada de píxeles de 30 m del centro-sur de Chile:

| Bloque | Fuente | Columnas |
|---|---|---|
| Trayectoria anual de cobertura 1999–2024 | MapBiomas Chile **Cobertura y Uso, Colección 2** | `lulc_1999` … `lulc_2024` |
| Historia de fuego 2013–2025 | MapBiomas **Fuego, Colección 1** (área quemada anual y tamaño de cicatriz) | `fire_first`, `fire_count`, `scar_size` |
| Covariables pre-incendio | SRTM, GRIP4 (caminos), cobertura MapBiomas 2012 | `elev`, `slope`, `aspect`, `dist_road`, `dist_urb`, `dist_agr`, `dist_silv` |
| Localización | FAO GAUL nivel 1 | `region`, `lon`, `lat` |

y una tabla de **hectáreas quemadas por región × año** para ponderar y escalar los resultados.

**Flujo:** (1) autenticación → (2) configuración → (3) carga y verificación de datos → (4) construcción del stack → (5) control visual opcional → (6) exportaciones a Google Drive.
Las exportaciones corren en los servidores de Earth Engine: lánzalas y revisa su avance en la pestaña *Tasks* del [Code Editor](https://code.earthengine.google.com/) o con `ee.batch.Task.list()`.


## 1. Autenticación

In [ ]:
# !pip install earthengine-api geemap  # descomenta si no los tienes
import re
import ee

PROYECTO_GEE = "TU-PROYECTO-GEE"   # <-- ID de tu proyecto de Google Cloud registrado en Earth Engine
try:
    ee.Initialize(project=PROYECTO_GEE)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROYECTO_GEE)
print("Earth Engine OK:", ee.String("listo").getInfo())

## 2. Configuración

In [ ]:
# --- Activos MapBiomas Chile, Cobertura Colección 2 (1999-2024) ---
# Se prueban en orden. Si ninguno funciona, copia el ID desde
# https://chile.mapbiomas.org/descargas/assets-en-google-earth-engine/ y agrégalo al inicio.
ASSETS_LULC = [
    "projects/mapbiomas-public/assets/chile/lulc/v1",
    "projects/mapbiomas-chile/assets/LULC/COLLECTION-02/CLASSIFICATIONS/classification-final/clasificacion-final-2",
]

# --- MapBiomas Fuego Colección 1 (2013-2025): GeoTIFF públicos en Google Cloud Storage ---
FUEGO_GCS = ("gs://mapbiomas-public/initiatives/chile/fire/collection1/"
             "{prod}_v1/mapbiomas_fire_chile_col1_{prod}_{y}.tif")
# Alternativa si loadGeoTIFF falla: sube los TIF como assets propios (celda 3.3) y usa este prefijo
FUEGO_ASSET_PREFIX = None   # p. ej. "projects/TU-PROYECTO-GEE/assets/fuego_col1"

ANIOS_LULC = list(range(1999, 2025))
ANIOS_FUEGO = list(range(2013, 2026))
COHORTES = list(range(2014, 2023))     # cohortes de incendio analizadas (k >= +2 observable)

# Zona de estudio: centro-sur mediterráneo y templado (Valparaíso a La Araucanía)
AOI = ee.Geometry.Rectangle([-74.0, -39.6, -70.0, -32.0])

N_TRAT_REGION = 15_000    # píxeles quemados por región
N_CTRL_REGION = 45_000    # píxeles nunca quemados por región (pool de controles)
SEMILLA = 2026
CARPETA_DRIVE = "premio_mapbiomas_fuego"
ESCALA = 30

## 3. Carga y verificación de datos
### 3.1 Cobertura y uso del suelo (Colección 2)

In [ ]:
def _bandas_por_anio(img):
    """Renombra bandas tipo 'classification_2012' / 'b2012' / '2012' a 'lulc_2012'."""
    bn = img.bandNames().getInfo()
    mapa = {}
    for b in bn:
        m = re.search(r"(19|20)\d{2}", b)
        if m and int(m.group()) in ANIOS_LULC:
            mapa[int(m.group())] = b
    faltan = [y for y in ANIOS_LULC if y not in mapa]
    if faltan:
        raise ValueError(f"faltan años {faltan[:5]}... bandas vistas: {bn[:6]}")
    return img.select([mapa[y] for y in ANIOS_LULC], [f"lulc_{y}" for y in ANIOS_LULC])


def cargar_lulc():
    for aid in ASSETS_LULC:
        # (a) imagen multibanda (una banda por año)
        try:
            img = _bandas_por_anio(ee.Image(aid))
            print("✔ cobertura cargada como Image:", aid)
            return img
        except Exception as e:
            err_img = e
        # (b) colección de imágenes (una por año, o una multibanda)
        try:
            col = ee.ImageCollection(aid)
            props = col.first().propertyNames().getInfo()
            if "year" in props:
                img = ee.Image.cat([col.filter(ee.Filter.eq("year", y)).mosaic().select(0)
                                    .rename(f"lulc_{y}") for y in ANIOS_LULC])
                img.bandNames().getInfo()
            else:
                img = _bandas_por_anio(col.mosaic())
            print("✔ cobertura cargada como ImageCollection:", aid)
            return img
        except Exception as e:
            print(f"✘ {aid}\n   Image: {str(err_img)[:120]}\n   Collection: {str(e)[:120]}")
    raise RuntimeError("No se pudo cargar la cobertura. Agrega el ID correcto a ASSETS_LULC.")


lulc = cargar_lulc().toInt16()
PROJ = lulc.projection().atScale(ESCALA)
# Verificación: clases presentes en un punto de la Región de Valparaíso
pt = ee.Geometry.Point([-71.45, -33.10])
print(lulc.select(["lulc_1999", "lulc_2012", "lulc_2024"]).reduceRegion(
    ee.Reducer.first(), pt, ESCALA).getInfo())

### 3.2 Fuego (Colección 1)
`annual_burned` > 0 = píxel quemado ese año (año calendario). `annual_burned_scar_size_range` = clase de tamaño de la cicatriz (1: <10 ha … 8: ≥100.000 ha).

In [ ]:
def fuego(prod, y):
    if FUEGO_ASSET_PREFIX:
        img = ee.Image(f"{FUEGO_ASSET_PREFIX}/{prod}_{y}")
    else:
        img = ee.Image.loadGeoTIFF(FUEGO_GCS.format(prod=prod, y=y))
    return img.select(0).unmask(0)

# Verificación: área quemada 2017 en un rectángulo del megaincendio de Las Máquinas (Maule)
rect = ee.Geometry.Rectangle([-72.3, -35.6, -72.0, -35.3])
try:
    ha_2017 = (fuego("annual_burned", 2017).gt(0).multiply(ee.Image.pixelArea()).divide(1e4)
               .reduceRegion(ee.Reducer.sum(), rect, ESCALA, maxPixels=1e10).getInfo())
    print("✔ Fuego OK. Hectáreas quemadas 2017 en el rectángulo de prueba:", ha_2017)
except Exception as e:
    print("✘ loadGeoTIFF falló:", str(e)[:300])
    print("  → usa la celda 3.3 para subir los GeoTIFF como assets y define FUEGO_ASSET_PREFIX.")

### 3.3 (Solo si 3.2 falló) Subir los GeoTIFF de Fuego como assets propios
Ejecuta en una terminal los comandos que imprime esta celda (requiere `earthengine` CLI autenticada). Earth Engine ingiere directo desde el bucket público, sin descargar nada a tu computador.

In [ ]:
prefijo = f"projects/{PROYECTO_GEE}/assets/fuego_col1"
print(f"earthengine create folder {prefijo}")
for prod in ["annual_burned", "annual_burned_scar_size_range"]:
    for y in ANIOS_FUEGO:
        print(f"earthengine upload image --asset_id={prefijo}/{prod}_{y} " + FUEGO_GCS.format(prod=prod, y=y))
# Al terminar las ingestas: FUEGO_ASSET_PREFIX = prefijo  y vuelve a correr 3.2

## 4. Construcción del stack de variables
### 4.1 Historia de fuego por píxel

In [ ]:
fire_first = ee.Image(0).toInt16()
fire_count = ee.Image(0).toInt16()
scar_size = ee.Image(0).toInt16()
for y in ANIOS_FUEGO:
    b = fuego("annual_burned", y).gt(0)
    nuevo = b.And(fire_first.eq(0))
    fire_first = fire_first.where(nuevo, y)
    fire_count = fire_count.add(b)
    scar_size = scar_size.where(nuevo, fuego("annual_burned_scar_size_range", y))
fuego_img = ee.Image.cat([fire_first.rename("fire_first"),
                          fire_count.rename("fire_count"),
                          scar_size.rename("scar_size")]).toInt16()

### 4.2 Covariables pre-incendio (fijadas en 2012, antes de toda cohorte)
Las distancias se calculan con `fastDistanceTransform` en una grilla de 120 m (UTM 19S) para cubrir hasta ~30 km; más allá se truncan. Usar 2012 evita controlar por variables afectadas por el propio fuego (*bad controls*).

In [ ]:
PROJ_DIST = ee.Projection("EPSG:32719").atScale(120)
VECINDAD = 256  # píxeles de 120 m -> ~30 km

def distancia(mascara):
    d = (mascara.selfMask().fastDistanceTransform(VECINDAD).sqrt()
         .multiply(120).reproject(PROJ_DIST))
    return d.unmask(VECINDAD * 120).toFloat()

l12 = lulc.select("lulc_2012")
dist_urb = distancia(l12.eq(24)).rename("dist_urb")
dist_agr = distancia(l12.eq(18).Or(l12.eq(15))).rename("dist_agr")
dist_silv = distancia(l12.eq(9)).rename("dist_silv")

try:
    caminos = (ee.FeatureCollection("projects/sat-io/open-datasets/GRIP4/South-America")
               .filterBounds(AOI))
    print("Tramos de camino GRIP4 en la zona:", caminos.size().getInfo())
    dist_road = distancia(ee.Image(0).byte().paint(caminos, 1)).rename("dist_road")
except Exception as e:
    print("⚠ GRIP4 no disponible; dist_road = -1 (el análisis la ignora).", str(e)[:120])
    dist_road = ee.Image(-1).toFloat().rename("dist_road")

dem = ee.Image("USGS/SRTMGL1_003")
terreno = ee.Terrain.products(dem).select(["elevation", "slope", "aspect"],
                                          ["elev", "slope", "aspect"]).toFloat()
lonlat = ee.Image.pixelLonLat().rename(["lon", "lat"])
covs = ee.Image.cat([terreno, dist_road, dist_urb, dist_agr, dist_silv, lonlat])

### 4.3 Estratos de muestreo
* **1 = quemado**: primer incendio detectado en una cohorte 2014–2022.
* **2 = nunca quemado** 2013–2025 (pool de controles).

Ambos restringidos a píxeles con cobertura 2012 natural o silvícola (bosque, matorral, pastizal, estepa, silvicultura), que es la población donde tiene sentido preguntar por *conversión*.

In [ ]:
ELEGIBLES_2012 = [3, 59, 60, 67, 66, 12, 63, 9]
elegible = l12.remap(ELEGIBLES_2012, [1] * len(ELEGIBLES_2012), 0).eq(1)
en_cohorte = fire_first.gte(min(COHORTES)).And(fire_first.lte(max(COHORTES)))
estrato = (ee.Image(0).where(en_cohorte, 1).where(fire_count.eq(0), 2)
           .updateMask(elegible).selfMask().rename("estrato").toInt8())

stack = ee.Image.cat([lulc, fuego_img, covs, estrato])
COLUMNAS = ([f"lulc_{y}" for y in ANIOS_LULC] +
            ["fire_first", "fire_count", "scar_size", "elev", "slope", "aspect",
             "dist_road", "dist_urb", "dist_agr", "dist_silv", "lon", "lat", "estrato", "region"])
print("Bandas del stack:", len(stack.bandNames().getInfo()))

### 4.4 Regiones (FAO GAUL nivel 1) dentro de la zona de estudio

In [ ]:
regiones = (ee.FeatureCollection("FAO/GAUL/2015/level1")
            .filter(ee.Filter.eq("ADM0_NAME", "Chile"))
            .filterBounds(AOI))
nombres = regiones.aggregate_array("ADM1_NAME").getInfo()
print(nombres)

def slug(s):
    return re.sub(r"[^a-zA-Z0-9]+", "_", s).strip("_").lower()

## 5. (Opcional) Control visual con geemap

In [ ]:
try:
    import geemap
    m = geemap.Map(center=[-35.4, -72.2], zoom=10)
    m.addLayer(lulc.select("lulc_2016"), {"min": 0, "max": 67}, "Cobertura 2016", False)
    m.addLayer(fire_first.selfMask(), {"min": 2013, "max": 2025,
               "palette": ["010079", "198D98", "CA6702", "750000"]}, "Año primer incendio")
    m.addLayer(estrato, {"min": 1, "max": 2, "palette": ["ff0000", "00aa00"]}, "Estrato", False)
    display(m)
except ImportError:
    print("geemap no instalado; omite este paso.")

## 6. Exportaciones a Google Drive
### 6.1 Muestra estratificada por región
Una tarea por región para evitar límites de memoria. Cada CSV trae una fila por píxel.

In [ ]:
tareas = []
for nombre in nombres:
    geom = regiones.filter(ee.Filter.eq("ADM1_NAME", nombre)).geometry().intersection(AOI, 100)
    muestra = (stack.stratifiedSample(
                   numPoints=0, classBand="estrato", region=geom, scale=ESCALA,
                   projection=PROJ, classValues=[1, 2],
                   classPoints=[N_TRAT_REGION, N_CTRL_REGION],
                   seed=SEMILLA, tileScale=16, geometries=False, dropNulls=True)
               .map(lambda f, n=nombre: f.set("region", n)))
    t = ee.batch.Export.table.toDrive(
        collection=muestra, description=f"muestra_{slug(nombre)}",
        folder=CARPETA_DRIVE, fileNamePrefix=f"muestra_{slug(nombre)}",
        fileFormat="CSV", selectors=COLUMNAS)
    t.start()
    tareas.append(t)
    print("lanzada:", nombre)

### 6.2 Hectáreas quemadas elegibles por región × cohorte
Sirve para (a) ponderar la muestra según el área quemada real y (b) traducir el efecto (ATT) a hectáreas atribuibles al fuego.

In [ ]:
area_ha = ee.Image.pixelArea().divide(1e4)
img_area = area_ha.addBands(fire_first.updateMask(en_cohorte).updateMask(elegible))

def tabla_region(nombre):
    geom = regiones.filter(ee.Filter.eq("ADM1_NAME", nombre)).geometry().intersection(AOI, 100)
    r = img_area.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName="cohorte"),
        geometry=geom, scale=ESCALA, crs=PROJ, maxPixels=1e13, tileScale=16)
    grupos = ee.List(r.get("groups"))
    return ee.FeatureCollection(grupos.map(
        lambda d: ee.Feature(None, ee.Dictionary(d)).set("region", nombre)))

areas = ee.FeatureCollection([tabla_region(n) for n in nombres]).flatten()
t = ee.batch.Export.table.toDrive(
    collection=areas, description="areas_quemadas", folder=CARPETA_DRIVE,
    fileNamePrefix="areas_quemadas", fileFormat="CSV",
    selectors=["region", "cohorte", "sum"])
t.start(); tareas.append(t)
print("lanzada: areas_quemadas")

### 6.3 Seguimiento de tareas

In [ ]:
import time
for t in tareas:
    s = t.status()
    print(f"{s['description']:40s} {s['state']}  {s.get('error_message', '')}")
# Cuando todas digan COMPLETED, descarga la carpeta de Drive a ./datos/ y abre el notebook 02.